# Methods Draft
## Exploring the Timescale Dependence of Magnetic Field Growth in Satellite Galaxies
### PHY 225-001 — Numerical Method  Project

---

## 1. Problem Setup

### 1.1 Scientific Context

Magnetic fields in galaxies are amplified by turbulent and large-scale dynamo processes driven by star formation, gas dynamics, and environmental interactions. Evidence from cosmological simulations suggests that **satellite galaxies** exhibit systematically stronger magnetic fields than central galaxies of the same stellar mass. However, the physical mechanism and timescale of this environmental amplification are not well understood.

This project uses data from the **IllustrisTNG cosmological MHD simulation** (TNG100-1) to track how the magnetic field strength of satellite galaxies evolves before and after the moment they first fall into their host halo (the **infall time**). A key challenge is that the simulation outputs are discrete in time and the resulting B-field time series are noisy, requiring robust smoothing before trends can be identified.

### 1.2 Simulation and Sample

IllustrisTNG is run with the AREPO moving-mesh MHD code. Relevant parameters for TNG100-1:

| Parameter | Value |
|---|---|
| Box size | $75 \, h^{-1}$ cMpc |
| Snapshots | 100 (z ≈ 20 → z = 0), non-uniformly spaced in time |
| Cosmology | Planck 2015: $H_0 = 67.74$ km/s/Mpc, $\Omega_m = 0.3089$, $h = 0.6774$ |
| Baryonic mass resolution | $\sim 1.4 \times 10^6 \, M_\odot$ |

**Sample selection:** Satellite subhalos at $z = 0$ (snapshot 99) with stellar mass $\log_{10}(M_*/M_\odot) \geq 9.5$, identified by `primary_flag = 0` in the SUBFIND group catalog.

**Magnetic field quantities** extracted from merger tree HDF5 files:
- `SubhaloBfldDisk` — mass-weighted B-field in the disk (comoving Gauss)
- `SubhaloBfldHalo` — mass-weighted B-field in the halo (comoving Gauss)

---
## 2. Time and Unit Conversions

### 2.1 Redshift → Lookback Time

All smoothing and analysis operations are performed in **lookback time** $t_{\rm lb}$ (Gyr) rather than redshift, since the snapshot outputs are non-uniformly spaced in redshift and operating directly on $z$ values distorts time-domain operations. The conversion uses the flat $\Lambda$CDM integral:

$$t_{\rm lb}(z) = \int_0^z \frac{dz'}{(1+z')\, H(z')}, \qquad H(z) = H_0\sqrt{\Omega_m(1+z)^3 + \Omega_\Lambda}$$

evaluated via `astropy.cosmology.FlatLambdaCDM`. A lookup table mapping snapshot number → $(z,\, a,\, t_{\rm lb})$ is built once at the start of the pipeline.

### 2.2 Comoving → Physical B-field

TNG stores magnetic fields in comoving Gauss. The physical field is:

$$B_{\rm phys} = \frac{B_{\rm com}}{a^2}, \qquad a = \frac{1}{1+z}$$

### 2.3 Time Coordinate Relative to Infall

After identifying the infall lookback time $t_{\rm lb,\,infall}$ for each galaxy (see Section 3), we define:

$$\Delta t = t_{\rm lb} - t_{\rm lb,\,infall}$$

so that $\Delta t > 0$ is before infall and $\Delta t < 0$ is after infall.

---
## 3. Merger Tree Analysis

### 3.1 Main Progenitor Branch

Each galaxy's evolutionary history is reconstructed from the **SubLink merger tree**, following the **Main Progenitor Branch (MPB)** — the chain of most-massive progenitors at each earlier snapshot. The MPB is downloaded as a cached HDF5 file via:
```
GET /api/TNG100-1/snapshots/99/subhalos/{id}/sublink/mpb.hdf5
```

### 3.2 Infall Time Detection

A galaxy is a **central** at snapshot $n$ if and only if its SubfindID equals the ID of the most massive subhalo in its host FoF group:

$$\text{is\_central}(n) = \mathbb{1}\left[\texttt{SubfindID}[n] = \texttt{GroupFirstSub}[n]\right]$$

The infall snapshot is identified by walking the MPB from $z=0$ backward and recording the last snapshot at which the galaxy was a central. The snapshot just before that (forward in time) is taken as the infall event:

```
last_central ← max { i : SubfindID[i] = GroupFirstSub[i] }
n_infall ← last_central − 1
```

---
## 4. Core Numerical Method: Kernel Smoothing (current redirection of the computational method)

### 4.1 Motivation

The raw B-field time series $\{(t_i, B_i)\}$ extracted from the merger tree is noisy, which reflects the discrete and stochastic nature of the simulation outputs. A standard **moving average** cannot be applied here because the snapshot times $t_i$ are **not uniformly spaced** in lookback time. Instead, we use **kernel smoothing** (also called kernel regression or the Nadaraya-Watson estimator), which naturally handles irregularly-sampled data.

### 4.2 Gaussian Kernel Smoother

Given the observed pairs $\{(t_i, B_i)\}_{i=1}^{n}$, the smoothed estimate at any query point $t$ is the **weighted average** of all observations, where the weight of observation $i$ decays with its distance from $t$:

$$\hat{B}(t) = \frac{\sum_{i=1}^{n} K_h(t - t_i)\, B_i}{\sum_{i=1}^{n} K_h(t - t_i)}$$

We use a **Gaussian kernel**:

$$K_h(u) = \exp\!\left(-\frac{u^2}{2h^2}\right)$$

where $h > 0$ is the **bandwidth** (in Gyr), which controls the degree of smoothing:
- Small $h$ → the smoother follows the data closely, preserving noise
- Large $h$ → the smoother is heavily averaged, potentially erasing real features

Selecting an appropriate $h$ is the central challenge.

### 4.3 Bandwidth Selection via Leave-One-Out Cross-Validation

We select the optimal bandwidth $h^*$ using **leave-one-out cross-validation (LOOCV)**. For each candidate bandwidth $h$, and for each observation index $j$, we:

1. Remove the $j$-th data point $(t_j, B_j)$ from the dataset
2. Compute the smoothed estimate at $t_j$ using the remaining $n-1$ points:

$$\hat{B}_{(-j)}(t_j) = \frac{\sum_{i \neq j} K_h(t_j - t_i)\, B_i}{\sum_{i \neq j} K_h(t_j - t_i)}$$

3. Record the squared prediction error $(B_j - \hat{B}_{(-j)}(t_j))^2$

The **LOOCV score** for bandwidth $h$ is the mean squared error over all left-out points:

$$\text{CV}(h) = \frac{1}{n} \sum_{j=1}^{n} \left( B_j - \hat{B}_{(-j)}(t_j) \right)^2$$

The optimal bandwidth is:

$$h^* = \arg\min_{h} \; \text{CV}(h)$$

found by evaluating $\text{CV}(h)$ over a grid of candidate bandwidths (e.g., $h \in [0.1, 3.0]$ Gyr) and selecting the minimum. A plot of $\text{CV}(h)$ vs. $h$ is produced to visualize the bias-variance tradeoff.

---
## 5. Ensemble Analysis

Once each galaxy's time series has been smoothed with its optimal bandwidth, the population-level trend is extracted by:

1. Aligning all smoothed tracks at $\Delta t = 0$ (infall time)
2. Evaluating $\hat{B}(\Delta t)$ on a common time grid for each galaxy
3. Computing the **binned median** across the ensemble:

$$\tilde{B}(\Delta t_k) = \text{median}\left\{ \hat{B}^{(i)}(\Delta t) : \Delta t \in \left[\Delta t_k - \tfrac{\delta}{2},\, \Delta t_k + \tfrac{\delta}{2}\right] \right\}$$

with bin width $\delta = 0.5$ Gyr and scatter characterized by the 16th/84th percentiles.

### Reach Goal: Exponential Growth Fit

If amplification post-infall is detected, we fit:

$$\hat{B}(\Delta t) = B_0 \, e^{-\Delta t / \tau}, \qquad \Delta t < 0$$

via nonlinear least squares (`scipy.optimize.curve_fit`), where $\tau$ is the amplification timescale in Gyr.

---
## 6. Implementation

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

# ─────────────────────────────────────────────────────────────────────────────
# KERNEL SMOOTHING FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def gaussian_kernel(u, h):
    """Gaussian kernel evaluated at distance u with bandwidth h."""
    return np.exp(-0.5 * (u / h)**2)


def kernel_smooth(t_query, t_obs, B_obs, h, exclude_idx=None):
    """
    Nadaraya-Watson kernel smoother with Gaussian kernel.

    Parameters
    ----------
    t_query    : float or array, query time point(s) in Gyr
    t_obs      : array, observed lookback times (Gyr)
    B_obs      : array, observed B-field values
    h          : float, bandwidth in Gyr
    exclude_idx: int or None, index to exclude (for LOOCV)

    Returns
    -------
    B_hat : smoothed estimate at t_query
    """
    mask = np.ones(len(t_obs), dtype=bool)
    if exclude_idx is not None:
        mask[exclude_idx] = False

    t_use = t_obs[mask]
    B_use = B_obs[mask]

    t_query = np.atleast_1d(t_query)
    B_hat   = np.zeros(len(t_query))

    for k, tq in enumerate(t_query):
        weights  = gaussian_kernel(tq - t_use, h)
        w_sum    = weights.sum()
        B_hat[k] = (weights @ B_use) / w_sum if w_sum > 0 else np.nan

    return B_hat if len(B_hat) > 1 else B_hat[0]


def loocv_score(h, t_obs, B_obs):
    """
    Leave-one-out cross-validation MSE for a given bandwidth h.

    For each observation j, fits the smoother on all OTHER points
    and measures the squared error at t_j.

    Parameters
    ----------
    h      : float, candidate bandwidth (Gyr)
    t_obs  : array, lookback times (Gyr)
    B_obs  : array, B-field values

    Returns
    -------
    cv_mse : float, mean squared LOOCV error
    """
    n      = len(t_obs)
    errors = np.zeros(n)
    for j in range(n):
        B_pred    = kernel_smooth(t_obs[j], t_obs, B_obs, h, exclude_idx=j)
        errors[j] = (B_obs[j] - B_pred)**2
    return errors.mean()


def select_bandwidth(t_obs, B_obs, h_grid=None):
    """
    Select optimal bandwidth via LOOCV over a grid of h values.

    Returns
    -------
    h_opt    : float, bandwidth minimizing LOOCV MSE
    h_grid   : array, candidate bandwidths
    cv_scores: array, LOOCV MSE at each h
    """
    if h_grid is None:
        h_grid = np.linspace(0.1, 3.0, 50)   # 0.1 to 3.0 Gyr

    cv_scores = np.array([loocv_score(h, t_obs, B_obs) for h in h_grid])
    h_opt     = h_grid[np.argmin(cv_scores)]
    return h_opt, h_grid, cv_scores


print("Kernel smoothing functions defined.")

Kernel smoothing functions defined.


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# APPLY TO A SINGLE GALAXY
# (assumes mpb_data and infall have already been computed in the analysis notebook)
# ─────────────────────────────────────────────────────────────────────────────

def smooth_galaxy(mpb_records, infall_info, field='B_disk_phys', h_grid=None):
    """
    Run kernel smoothing + LOOCV bandwidth selection for one galaxy.

    Parameters
    ----------
    mpb_records  : list of dicts from get_mpb() + compute_B_physical()
    infall_info  : dict from find_infall_snap(), or None
    field        : 'B_disk_phys' or 'B_halo_phys'
    h_grid       : optional array of candidate bandwidths in Gyr

    Returns
    -------
    result dict with keys: t_obs, B_obs, h_opt, h_grid, cv_scores,
                           t_fine, B_smooth, dt_infall (if infall found)
    """
    # Extract valid data points and sort by lookback time
    valid = [r for r in mpb_records
             if np.isfinite(r.get(field, np.nan)) and r[field] > 0]
    if len(valid) < 5:
        return None

    t_obs = np.array([r['lookback_time'] for r in valid])
    B_obs = np.array([r[field]           for r in valid])
    order = np.argsort(t_obs)
    t_obs, B_obs = t_obs[order], B_obs[order]

    # Bandwidth selection
    h_opt, h_grid_out, cv_scores = select_bandwidth(t_obs, B_obs, h_grid)

    # Smooth on a fine time grid
    t_fine   = np.linspace(t_obs.min(), t_obs.max(), 300)
    B_smooth = kernel_smooth(t_fine, t_obs, B_obs, h_opt)

    result = {
        't_obs'     : t_obs,
        'B_obs'     : B_obs,
        'h_opt'     : h_opt,
        'h_grid'    : h_grid_out,
        'cv_scores' : cv_scores,
        't_fine'    : t_fine,
        'B_smooth'  : B_smooth,
    }
    if infall_info:
        result['t_infall'] = infall_info['lookback_time']
        result['dt_obs']   = t_obs  - infall_info['lookback_time']
        result['dt_fine']  = t_fine - infall_info['lookback_time']

    return result


# ── Run on the test galaxy ────────────────────────────────────────────────────
# (mpb_data and infall must be defined from the main analysis notebook cells)
smooth_result = smooth_galaxy(mpb_data, infall)

if smooth_result:
    print(f"Optimal bandwidth : h* = {smooth_result['h_opt']:.3f} Gyr")
    print(f"Data points used  : {len(smooth_result['t_obs'])}")
else:
    print("Not enough valid data points for smoothing.")

NameError: name 'mpb_data' is not defined

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DIAGNOSTIC PLOTS: LOOCV curve + raw vs smoothed B-field
# ─────────────────────────────────────────────────────────────────────────────

def plot_smoothing_diagnostics(smooth_result, subhalo_id=None):
    """
    Two-panel plot:
      Left : LOOCV MSE vs bandwidth h (with optimal h marked)
      Right: Raw data vs smoothed curve, with infall marked
    """
    if smooth_result is None:
        print("No smoothing result to plot.")
        return

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    # ── Left: LOOCV curve ────────────────────────────────────────────────────
    ax1.plot(smooth_result['h_grid'], smooth_result['cv_scores'],
             color='steelblue', lw=2)
    ax1.axvline(smooth_result['h_opt'], color='tomato', lw=2, ls='--',
                label=f"$h^* = {smooth_result['h_opt']:.2f}$ Gyr")
    ax1.set_xlabel('Bandwidth $h$ [Gyr]', fontsize=12)
    ax1.set_ylabel('LOOCV MSE', fontsize=12)
    ax1.set_title('Bandwidth Selection via LOOCV', fontsize=13)
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)

    # ── Right: Raw vs smoothed ────────────────────────────────────────────────
    # Use dt (time relative to infall) if available, otherwise lookback time
    if 'dt_obs' in smooth_result:
        x_obs  = smooth_result['dt_obs']
        x_fine = smooth_result['dt_fine']
        xlabel = r'$\Delta t = t_{\rm lb} - t_{\rm infall}$ [Gyr]'
        ax2.axvline(0, color='black', lw=1.5, ls=':', label='Infall')
    else:
        x_obs  = smooth_result['t_obs']
        x_fine = smooth_result['t_fine']
        xlabel = 'Lookback time [Gyr]'

    ax2.scatter(x_obs, smooth_result['B_obs'],
                s=25, color='steelblue', alpha=0.6, zorder=3, label='Raw data')
    ax2.plot(x_fine, smooth_result['B_smooth'],
             color='tomato', lw=2.5,
             label=f'Kernel smoother ($h^*={smooth_result["h_opt"]:.2f}$ Gyr)')

    ax2.set_yscale('log')
    ax2.set_xlabel(xlabel, fontsize=12)
    ax2.set_ylabel(r'$B_{\rm phys}$ [G]', fontsize=12)
    title = f'B-field: Raw vs Smoothed'
    if subhalo_id:
        title += f' — Subhalo {subhalo_id}'
    ax2.set_title(title, fontsize=13)
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('results/figures/kernel_smoothing_diagnostics.png', dpi=150)
    plt.show()


plot_smoothing_diagnostics(smooth_result, subhalo_id=test_id)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# APPLY SMOOTHING TO THE FULL GALAXY SAMPLE
# ─────────────────────────────────────────────────────────────────────────────

def smooth_all_galaxies(all_galaxies, field='B_disk_phys'):
    """
    Apply kernel smoothing to each galaxy in the sample.
    Returns the list with 'smooth' key added to each entry.
    """
    for i, gal in enumerate(all_galaxies):
        print(f"[{i+1}/{len(all_galaxies)}] Smoothing SubhaloID {gal['subhalo_id']} ...", end=' ')
        result = smooth_galaxy(gal['mpb'], gal['infall'], field=field)
        gal['smooth'] = result
        if result:
            print(f"h* = {result['h_opt']:.2f} Gyr")
        else:
            print("skipped (insufficient data)")
    return all_galaxies


all_galaxies = smooth_all_galaxies(all_galaxies)

n_smoothed = sum(1 for g in all_galaxies if g['smooth'] is not None)
h_vals     = [g['smooth']['h_opt'] for g in all_galaxies if g['smooth']]
print(f"\nSuccessfully smoothed : {n_smoothed} galaxies")
print(f"Optimal bandwidth     : median h* = {np.median(h_vals):.2f} Gyr  "
      f"(range {min(h_vals):.2f}–{max(h_vals):.2f} Gyr)")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ENSEMBLE PLOT: Smoothed B-field vs Time Since Infall
# ─────────────────────────────────────────────────────────────────────────────

def plot_smoothed_ensemble(all_galaxies, dt_range=(-4, 6), bin_width=0.5):
    """
    Overlay smoothed B-field tracks and plot the binned median.
    """
    fig, ax = plt.subplots(figsize=(11, 5))

    # Collect smoothed (dt, B) pairs for binning
    all_dt, all_B = [], []

    for gal in all_galaxies:
        s = gal['smooth']
        if s is None or 'dt_fine' not in s:
            continue

        # Trim to dt_range for display
        mask = (s['dt_fine'] >= dt_range[0]) & (s['dt_fine'] <= dt_range[1])
        ax.plot(s['dt_fine'][mask], s['B_smooth'][mask],
                color='steelblue', alpha=0.2, lw=1)

        all_dt.extend(s['dt_fine'][mask].tolist())
        all_B.extend(s['B_smooth'][mask].tolist())

    all_dt = np.array(all_dt)
    all_B  = np.array(all_B)

    # Binned median + percentiles
    bins   = np.arange(dt_range[0], dt_range[1] + bin_width, bin_width)
    bin_c  = 0.5 * (bins[:-1] + bins[1:])
    median, p16, p84 = [], [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (all_dt >= lo) & (all_dt < hi)
        if mask.sum() > 2:
            vals = all_B[mask]
            median.append(np.median(vals))
            p16.append(np.percentile(vals, 16))
            p84.append(np.percentile(vals, 84))
        else:
            median.append(np.nan); p16.append(np.nan); p84.append(np.nan)

    median = np.array(median)
    p16, p84 = np.array(p16), np.array(p84)
    valid = np.isfinite(median)

    ax.fill_between(bin_c[valid], p16[valid], p84[valid],
                    alpha=0.4, color='tomato', label='16th–84th percentile')
    ax.plot(bin_c[valid], median[valid],
            color='tomato', lw=2.5, label='Median (smoothed)')
    ax.axvline(0, color='black', lw=2, ls='--', label='Infall ($\\Delta t=0$)')

    ax.set_yscale('log')
    ax.set_xlabel(r'$\Delta t = t_{\rm lb} - t_{\rm infall}$ [Gyr]', fontsize=13)
    ax.set_ylabel(r'$B_{\rm phys}$ [G]', fontsize=13)
    ax.set_title('Smoothed B-field vs Time Since Infall — TNG100 Satellites', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('results/figures/B_smoothed_vs_dt_infall.png', dpi=150)
    plt.show()


plot_smoothed_ensemble(all_galaxies)